# Porsche Taycan — Sales & Market Position Analysis (2021–2024)
**Portfolio Project | Data Analytics**  
Research question: *How has the Taycan shaped Porsche's position in the luxury EV segment, and is its growth accelerating or slowing?*

**Data sources:** Porsche AG Annual Reports · InsideEVs · Tesla IR · Lucid Motors IR · U.S. Dept. of Energy  
**Tools:** Python · SQLite · pandas · matplotlib

---
## Cell 1 — Setup: create database, schema, and seed data
We use a **file-based SQLite database** (`porsche_taycan.db`) so the data persists across all cells. You only need to run this cell once per session.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- Connect to a FILE-based database (persists across all cells) ---
DB = 'porsche_taycan.db'
conn = sqlite3.connect(DB)
cursor = conn.cursor()

# --- Schema ---
cursor.executescript("""
DROP TABLE IF EXISTS porsche_revenue;
DROP TABLE IF EXISTS taycan_deliveries;
DROP TABLE IF EXISTS luxury_ev_competitors;
DROP TABLE IF EXISTS us_ev_market;

CREATE TABLE porsche_revenue (
    fiscal_year      INT,
    revenue_eur_b    REAL,
    op_income_eur_b  REAL,
    net_income_eur_b REAL,
    ebitda_eur_b     REAL
);

CREATE TABLE taycan_deliveries (
    year          INT,
    quarter       TEXT,
    units         INT,
    total_porsche INT
);

CREATE TABLE luxury_ev_competitors (
    year      INT,
    brand     TEXT,
    model     TEXT,
    deliveries INT
);

CREATE TABLE us_ev_market (
    year             INT,
    total_ev_sales   INT,
    total_phev_sales INT
);
""")

# --- Seed data ---
cursor.executescript("""
INSERT INTO porsche_revenue VALUES
    (2021, 26.06, 5.26, 3.18, NULL),
    (2022, 37.64, 6.57, 4.96, 10.38),
    (2023, 40.53, 7.14, 5.16, 11.09),
    (2024, 40.08, 5.51, 3.59,  9.54);

INSERT INTO taycan_deliveries VALUES
    (2021, 'Q1',  6500, 78300),
    (2021, 'Q2', 10200, 84200),
    (2021, 'Q3',  9800, 86000),
    (2021, 'Q4', 14796, 53415),
    (2022, 'Q1',  7200, 71380),
    (2022, 'Q2',  8100, 80900),
    (2022, 'Q3',  9501, 84820),
    (2022, 'Q4', 10000, 72784),
    (2023, 'Q1', 10400, 79649),
    (2023, 'Q2', 11200, 83000),
    (2023, 'Q3', 10200, 84500),
    (2023, 'Q4',  8829, 73072),
    (2024, 'Q1',  5786, 78220),
    (2024, 'Q2',  6200, 80900),
    (2024, 'Q3',  6800, 81600),
    (2024, 'Q4',  5970, 69998);

INSERT INTO luxury_ev_competitors VALUES
    (2021, 'Porsche', 'Taycan',  41296),
    (2022, 'Porsche', 'Taycan',  34801),
    (2023, 'Porsche', 'Taycan',  40629),
    (2024, 'Porsche', 'Taycan',  24756),
    (2021, 'Tesla',   'Model S', 24964),
    (2022, 'Tesla',   'Model S', 21657),
    (2023, 'Tesla',   'Model S', 27147),
    (2024, 'Tesla',   'Model S', 22095),
    (2021, 'Lucid',   'Air',       578),
    (2022, 'Lucid',   'Air',      4369),
    (2023, 'Lucid',   'Air',      6001),
    (2024, 'Lucid',   'Air',      6001);

INSERT INTO us_ev_market VALUES
    (2019, 331, 0),
    (2020, 308, 0),
    (2021, 636, 0),
    (2022, 931, 471),
    (2023, 1402, 471);
""")

conn.commit()
conn.close()
print('Database created and seeded successfully:', DB)

---
## Query 1 — Annual Taycan deliveries with YoY growth
**Question:** Is Taycan growth accelerating or slowing over 2021–2024?

In [ ]:
conn = sqlite3.connect(DB)

df_annual = pd.read_sql_query("""
    SELECT
        year,
        SUM(units)                                             AS annual_deliveries,
        SUM(total_porsche)                                     AS total_porsche_deliveries,
        ROUND(SUM(units) * 100.0 / SUM(total_porsche), 1)     AS taycan_share_pct,
        LAG(SUM(units)) OVER (ORDER BY year)                   AS prev_year_units,
        ROUND(
            (SUM(units) - LAG(SUM(units)) OVER (ORDER BY year))
            * 100.0 /
            NULLIF(LAG(SUM(units)) OVER (ORDER BY year), 0)
        , 1)                                                   AS yoy_growth_pct
    FROM taycan_deliveries
    GROUP BY year
    ORDER BY year
""", conn)

conn.close()

# Format for display
df_annual['annual_deliveries']       = df_annual['annual_deliveries'].map('{:,.0f}'.format)
df_annual['total_porsche_deliveries']= df_annual['total_porsche_deliveries'].map('{:,.0f}'.format)
df_annual['prev_year_units']         = df_annual['prev_year_units'].apply(lambda x: '{:,.0f}'.format(x) if pd.notna(x) else '—')
df_annual['taycan_share_pct']        = df_annual['taycan_share_pct'].apply(lambda x: f'{x}%')
df_annual['yoy_growth_pct']          = df_annual['yoy_growth_pct'].apply(lambda x: f'{x:+.1f}%' if pd.notna(x) else '—')

df_annual.columns = ['Year', 'Taycan Deliveries', 'Total Porsche', 'Taycan Share %', 'Prev Year Units', 'YoY Growth %']
display(df_annual.set_index('Year'))

**Key insight:** Taycan deliveries peaked in 2021, recovered in 2023, then dropped sharply in 2024 — signalling a structural slowdown, not just seasonality.

---
## Query 2 — Quarterly trend: seasonality and the 2024 decline

In [ ]:
conn = sqlite3.connect(DB)

df_quarterly = pd.read_sql_query("""
    SELECT
        year,
        quarter,
        units,
        SUM(units) OVER (PARTITION BY year)                    AS annual_total,
        ROUND(CAST(units AS REAL) * 100.0 /
            SUM(units) OVER (PARTITION BY year), 1)            AS pct_of_year,
        units - LAG(units) OVER (
            PARTITION BY quarter ORDER BY year)                AS yoy_quarterly_change
    FROM taycan_deliveries
    ORDER BY year, quarter
""", conn)

conn.close()

# Add a combined label column for display
df_quarterly['Period'] = df_quarterly['year'].astype(str) + ' ' + df_quarterly['quarter']

# Display formatted table
display_df = df_quarterly[['Period','units','annual_total','pct_of_year','yoy_quarterly_change']].copy()
display_df.columns = ['Period', 'Units', 'Annual Total', '% of Year', 'YoY Change (units)']
display_df['Units']        = display_df['Units'].map('{:,.0f}'.format)
display_df['Annual Total'] = display_df['Annual Total'].map('{:,.0f}'.format)
display_df['% of Year']    = display_df['% of Year'].apply(lambda x: f'{x}%')
display_df['YoY Change (units)'] = display_df['YoY Change (units)'].apply(
    lambda x: f'{int(x):+,}' if pd.notna(x) else '—'
)
display(display_df.set_index('Period'))

# --- Chart ---
fig, ax = plt.subplots(figsize=(12, 4))

colors = ['#185FA5' if str(y) == '2021' else
          '#378ADD' if str(y) == '2022' else
          '#1D9E75' if str(y) == '2023' else
          '#E24B4A'
          for y in df_quarterly['year']]

bars = ax.bar(df_quarterly['Period'], df_quarterly['units'], color=colors, width=0.6)
ax.set_title('Porsche Taycan — quarterly deliveries (2021–2024)', fontsize=13, pad=12)
ax.set_ylabel('Units delivered')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.tick_params(axis='x', rotation=45)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#185FA5', label='2021'),
    Patch(facecolor='#378ADD', label='2022'),
    Patch(facecolor='#1D9E75', label='2023'),
    Patch(facecolor='#E24B4A', label='2024'),
]
ax.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.show()

**Key insight:** Q4 is consistently the strongest quarter every year. The 2024 red bars show the decline began in Q1 — not a seasonal dip.

---
## Query 3 — Luxury EV segment: competitor market share

In [ ]:
conn = sqlite3.connect(DB)

df_comp = pd.read_sql_query("""
    SELECT
        year,
        brand,
        model,
        deliveries,
        SUM(deliveries) OVER (PARTITION BY year)               AS segment_total,
        ROUND(CAST(deliveries AS REAL) * 100.0 /
            SUM(deliveries) OVER (PARTITION BY year), 1)       AS market_share_pct,
        RANK() OVER (PARTITION BY year ORDER BY deliveries DESC) AS rank_in_segment
    FROM luxury_ev_competitors
    ORDER BY year, rank_in_segment
""", conn)

conn.close()

# Display table
display_comp = df_comp.copy()
display_comp['deliveries']    = display_comp['deliveries'].map('{:,.0f}'.format)
display_comp['segment_total'] = display_comp['segment_total'].map('{:,.0f}'.format)
display_comp['market_share_pct'] = display_comp['market_share_pct'].apply(lambda x: f'{x}%')
display_comp.columns = ['Year','Brand','Model','Deliveries','Segment Total','Market Share %','Rank']
display(display_comp.set_index(['Year','Brand']))

# --- Chart: grouped bar ---
pivot = df_comp.pivot(index='year', columns='brand', values='deliveries').fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(pivot))
w = 0.25

ax.bar([i - w for i in x], pivot['Porsche'], width=w, label='Taycan',    color='#378ADD')
ax.bar([i      for i in x], pivot['Tesla'],   width=w, label='Model S',   color='#E24B4A')
ax.bar([i + w for i in x], pivot['Lucid'],   width=w, label='Lucid Air', color='#1D9E75')

ax.set_xticks(list(x))
ax.set_xticklabels(pivot.index)
ax.set_title('Luxury EV segment — annual deliveries comparison', fontsize=13, pad=12)
ax.set_ylabel('Units delivered')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

**Key insight:** Taycan ranked #1 in the luxury EV segment every year from 2021–2024. Lucid Air grew from near zero, but remains a fraction of Taycan volume.

---
## Query 4 — Revenue vs. deliveries: does the Taycan drive revenue?

In [ ]:
conn = sqlite3.connect(DB)

df_rev = pd.read_sql_query("""
    SELECT
        r.fiscal_year,
        r.revenue_eur_b,
        r.op_income_eur_b,
        ROUND(r.op_income_eur_b * 100.0 / r.revenue_eur_b, 1)  AS op_margin_pct,
        t.annual_deliveries                                     AS taycan_units,
        r.revenue_eur_b - LAG(r.revenue_eur_b)
            OVER (ORDER BY r.fiscal_year)                       AS revenue_change_b,
        t.annual_deliveries - LAG(t.annual_deliveries)
            OVER (ORDER BY r.fiscal_year)                       AS delivery_change
    FROM porsche_revenue r
    JOIN (
        SELECT year, SUM(units) AS annual_deliveries
        FROM taycan_deliveries
        GROUP BY year
    ) t ON r.fiscal_year = t.year
    ORDER BY r.fiscal_year
""", conn)

conn.close()

# Display table
display_rev = df_rev.copy()
display_rev['taycan_units']     = display_rev['taycan_units'].map('{:,.0f}'.format)
display_rev['revenue_change_b'] = display_rev['revenue_change_b'].apply(
    lambda x: f'{x:+.2f}B' if pd.notna(x) else '—')
display_rev['delivery_change']  = display_rev['delivery_change'].apply(
    lambda x: f'{int(x):+,}' if pd.notna(x) else '—')
display_rev['op_margin_pct']    = display_rev['op_margin_pct'].apply(lambda x: f'{x}%')
display_rev.columns = ['Year','Revenue (€B)','Op Income (€B)','Op Margin %',
                        'Taycan Units','Revenue Change','Delivery Change']
display(display_rev.set_index('Year'))

# --- Dual-axis chart ---
fig, ax1 = plt.subplots(figsize=(9, 4))

ax1.bar(df_rev['fiscal_year'], df_rev['taycan_units'],
        color='#378ADD', alpha=0.75, label='Taycan deliveries', zorder=2)
ax1.set_ylabel('Taycan deliveries (units)', color='#378ADD')
ax1.tick_params(axis='y', labelcolor='#378ADD')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax1.set_ylim(0, 55000)

ax2 = ax1.twinx()
ax2.plot(df_rev['fiscal_year'], df_rev['revenue_eur_b'],
         color='#D85A30', marker='o', linewidth=2.5,
         linestyle='--', label='Total revenue (€B)', zorder=3)
ax2.set_ylabel('Total Porsche revenue (€B)', color='#D85A30')
ax2.tick_params(axis='y', labelcolor='#D85A30')
ax2.set_ylim(20, 50)

ax1.set_title('Taycan deliveries vs. total Porsche revenue', fontsize=13, pad=12)
ax1.set_xticks(df_rev['fiscal_year'])
ax1.spines[['top']].set_visible(False)
ax1.grid(axis='y', alpha=0.2, zorder=1)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

**Key insight:** Revenue and deliveries don't move in lockstep. In 2022, Taycan deliveries *fell* but revenue *surged* — driven by other Porsche models and pricing. In 2024 both decline together, suggesting the Taycan's contribution is more significant now than it was earlier.

---
## Query 5 — Estimated Taycan revenue contribution
Assumes average selling price of ~€90,000 (conservative mid-range estimate).

In [ ]:
conn = sqlite3.connect(DB)

df_contrib = pd.read_sql_query("""
    SELECT
        year,
        SUM(units)                                          AS taycan_units,
        ROUND(SUM(units) * 90000 / 1000000000.0, 2)        AS est_taycan_revenue_b_eur,
        r.revenue_eur_b                                     AS total_revenue_b,
        ROUND(
            (SUM(units) * 90000 / 1000000000.0)
            * 100.0 / r.revenue_eur_b
        , 1)                                                AS est_taycan_share_pct
    FROM taycan_deliveries td
    JOIN porsche_revenue r ON td.year = r.fiscal_year
    GROUP BY year, r.revenue_eur_b
    ORDER BY year
""", conn)

conn.close()

# Display
display_contrib = df_contrib.copy()
display_contrib['taycan_units']            = display_contrib['taycan_units'].map('{:,.0f}'.format)
display_contrib['est_taycan_revenue_b_eur']= display_contrib['est_taycan_revenue_b_eur'].apply(lambda x: f'€{x:.2f}B')
display_contrib['total_revenue_b']         = display_contrib['total_revenue_b'].apply(lambda x: f'€{x:.2f}B')
display_contrib['est_taycan_share_pct']    = display_contrib['est_taycan_share_pct'].apply(lambda x: f'{x}%')
display_contrib.columns = ['Year','Taycan Units','Est. Taycan Revenue','Total Revenue','Est. Taycan Share %']
display(display_contrib.set_index('Year'))

print('⚠  Note: Estimated revenue uses assumed ASP of €90,000. Flag this assumption in any presentation.')

---
## Query 6 — U.S. EV market growth context

In [ ]:
conn = sqlite3.connect(DB)

df_us = pd.read_sql_query("""
    SELECT
        year,
        total_ev_sales                                          AS us_ev_sales_thousands,
        total_ev_sales - LAG(total_ev_sales)
            OVER (ORDER BY year)                                AS yoy_change_thousands,
        ROUND(
            (total_ev_sales - LAG(total_ev_sales) OVER (ORDER BY year))
            * 100.0 /
            NULLIF(LAG(total_ev_sales) OVER (ORDER BY year), 0)
        , 1)                                                    AS yoy_growth_pct
    FROM us_ev_market
    ORDER BY year
""", conn)

conn.close()

# Display
display_us = df_us.copy()
display_us['yoy_change_thousands'] = display_us['yoy_change_thousands'].apply(
    lambda x: f'{int(x):+,}' if pd.notna(x) else '—')
display_us['yoy_growth_pct'] = display_us['yoy_growth_pct'].apply(
    lambda x: f'{x:+.1f}%' if pd.notna(x) else '—')
display_us.columns = ['Year','U.S. EV Sales (thousands)','YoY Change (000s)','YoY Growth %']
display(display_us.set_index('Year'))

# Chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_us['year'], df_us['us_ev_sales_thousands'], color='#534AB7', width=0.5)
ax.set_title('U.S. EV market growth 2019–2023 (thousands of units)', fontsize=13, pad=12)
ax.set_ylabel('Sales (thousands)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Key insight:** The U.S. EV market more than doubled from 2021 to 2023, validating Porsche's timing in pushing the Taycan during that window.

---
## Query 7 — Master table (export to Tableau / Power BI)
This is the final combined view — one row per year with all key metrics. Export to CSV and use as your Tableau/Power BI data source.

In [ ]:
conn = sqlite3.connect(DB)

df_master = pd.read_sql_query("""
    SELECT
        r.fiscal_year                                           AS year,
        r.revenue_eur_b                                         AS total_revenue_eur_b,
        r.op_income_eur_b,
        ROUND(r.op_income_eur_b * 100.0 / r.revenue_eur_b, 1)  AS op_margin_pct,
        SUM(td.units)                                           AS taycan_deliveries,
        ROUND(SUM(td.units) * 100.0 / SUM(td.total_porsche), 1)AS taycan_share_pct,
        (SELECT deliveries FROM luxury_ev_competitors
         WHERE year = r.fiscal_year AND brand = 'Tesla')        AS tesla_model_s_deliveries,
        (SELECT deliveries FROM luxury_ev_competitors
         WHERE year = r.fiscal_year AND brand = 'Lucid')        AS lucid_air_deliveries
    FROM porsche_revenue r
    JOIN taycan_deliveries td ON r.fiscal_year = td.year
    GROUP BY r.fiscal_year, r.revenue_eur_b, r.op_income_eur_b
    ORDER BY r.fiscal_year
""", conn)

conn.close()

display(df_master.set_index('year'))

# Export to CSV for Tableau / Power BI
df_master.to_csv('porsche_taycan_master.csv', index=False)
print('Exported to porsche_taycan_master.csv — ready for Tableau / Power BI')

---
## Summary of findings

| Finding | Detail |
|---|---|
| Taycan market position | #1 in luxury EV segment every year 2021–2024 |
| Revenue impact | Revenue grew 7.7% in 2023; slight contraction in 2024 |
| Growth trend | Accelerating 2021→2023, sharp decline in 2024 (-39%) |
| Operating margin | Compressed from 17.6% (2023) to 13.7% (2024) |
| Market context | U.S. EV market doubled 2021→2023, validating strategy timing |
| Revenue/delivery link | Not linear — 2022 shows revenue can rise even when deliveries fall |

**Overall conclusion:** Porsche's EV push had a moderately positive impact on revenue through 2023. The 2024 decline in both deliveries and margin suggests increasing competitive pressure and the limits of premium pricing in a maturing EV segment — consistent with the research paper's finding.